<a href="https://colab.research.google.com/github/prasadboyina/Gen-AI/blob/main/Seq2SeqModel(24_06_2026).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sequence to Sequence Model**

# **Enocder:**

The Encoder is a type of Neural Network(RNN/LSTM/GRU) that reads the input sentences word by word and compress all the information into **Context Vector**(we also calles it as hidden/thought vecotor).

# **Decoder:**

Now, the decoder is another neural network that takes the context vector from encoder and generate the transalted Sentence(One word at a time).

In Seq2Seq Model--> We have three special tokens:

* '<SOS>': Start of Sentence(It tells the decoder that we are now starting the sentence)-->(index-1)

* '<EOS>': End of Sequence(It tells the decoder that the sentece is completed)(index-2)

* '<PAD>'; pad token is responsible to match the length of batches in input(index-0)

**Import the Libraries**

In [ ]:
import tensorflow as tf
import numpy as np

**We will prepare the training Data**

In [ ]:
data = [
    ('I am happy', 'मैं खुश हूँ'),
    ('You are beautiful', 'तुम सुंदर हो'),
    ('We are learning', 'हम सीख रहे हैं'),
    ('He is sleeping', 'वह सो रहा है'),
    ('She is singing', 'वह गा रही है'),
    ('They are coming', 'वे आ रहे हैं'),
    ('It is raining', 'बारिश हो रही है'),
    ('Who are you ?', 'आप कौन हैं ?'),
    ('Where are we ?', 'हम कहाँ हैं ?'),
    ('What is this ?', 'यह क्या है ?')
]

**We will build the vocab.**

In [ ]:
def build_vocab(sentences):
  vocab={'<PAD>':0,'<SOS>':1,'<EOS>':2}
  next_index=3
  for sentence in sentences:
    words=sentence.split()

    for word in words:
      if word not in vocab:
        vocab[word]=next_index
        next_index+=1
  return vocab

In [ ]:
for pair in data:
  print(pair[1])

मैं खुश हूँ
तुम सुंदर हो
हम सीख रहे हैं
वह सो रहा है
वह गा रही है
वे आ रहे हैं
बारिश हो रही है
आप कौन हैं ?
हम कहाँ हैं ?
यह क्या है ?


In [ ]:
english_sentences=[pair[0] for pair in data]
english_sentences

['I am happy',
 'You are beautiful',
 'We are learning',
 'He is sleeping',
 'She is singing',
 'They are coming',
 'It is raining',
 'Who are you ?',
 'Where are we ?',
 'What is this ?']

In [ ]:
hindi_sentences=[pair[1] for pair in data]
hindi_sentences

['मैं खुश हूँ',
 'तुम सुंदर हो',
 'हम सीख रहे हैं',
 'वह सो रहा है',
 'वह गा रही है',
 'वे आ रहे हैं',
 'बारिश हो रही है',
 'आप कौन हैं ?',
 'हम कहाँ हैं ?',
 'यह क्या है ?']

In [ ]:
eng_vocab=build_vocab(english_sentences)
hin_vocab=build_vocab(hindi_sentences)

In [ ]:
eng_vocab

{'<PAD>': 0,
 '<SOS>': 1,
 '<EOS>': 2,
 'I': 3,
 'am': 4,
 'happy': 5,
 'You': 6,
 'are': 7,
 'beautiful': 8,
 'We': 9,
 'learning': 10,
 'He': 11,
 'is': 12,
 'sleeping': 13,
 'She': 14,
 'singing': 15,
 'They': 16,
 'coming': 17,
 'It': 18,
 'raining': 19,
 'Who': 20,
 'you': 21,
 '?': 22,
 'Where': 23,
 'we': 24,
 'What': 25,
 'this': 26}

In [ ]:
hin_vocab

{'<PAD>': 0,
 '<SOS>': 1,
 '<EOS>': 2,
 'मैं': 3,
 'खुश': 4,
 'हूँ': 5,
 'तुम': 6,
 'सुंदर': 7,
 'हो': 8,
 'हम': 9,
 'सीख': 10,
 'रहे': 11,
 'हैं': 12,
 'वह': 13,
 'सो': 14,
 'रहा': 15,
 'है': 16,
 'गा': 17,
 'रही': 18,
 'वे': 19,
 'आ': 20,
 'बारिश': 21,
 'आप': 22,
 'कौन': 23,
 '?': 24,
 'कहाँ': 25,
 'यह': 26,
 'क्या': 27}

# **Let's Convert Sentences to Number(Toeknization)**

In [ ]:
def sentence_to_number(sentence,vocab):
  #Split the sentence into words
  words=sentence.split()
  #It will convert each word to its number(use 0 if word not found)
  numbers=[vocab.get(word,0) for word in words]

  #Add SOS and EOS
  numbers=[vocab['<SOS>']]+ numbers+ [vocab['<EOS>']]

  return numbers

In [ ]:
test_sentence='बारिश हो रही है'
test_number=sentence_to_number(test_sentence,hin_vocab)

In [ ]:
test_number

[1, 21, 8, 18, 16, 2]

In [ ]:
test_sentence='मैं खुश हूँ'
test_number=sentence_to_number(test_sentence,hin_vocab)

In [ ]:
test_number

[1, 3, 4, 5, 2]

**Now we need to prepare data with equal length using padding.**

In [ ]:
def prepare_data(data,eng_vocab,hin_vocab):
  #Let's convert each english sentence to number
  src_number=[sentence_to_number(pair[0],eng_vocab) for pair in data]
  tgt_number=[sentence_to_number(pair[1],hin_vocab) for pair in data]

  #we Pad in Eng sent
  src_padded=tf.keras.preprocessing.sequence.pad_sequences(
      src_number,
      padding='post',
      value=eng_vocab['<PAD>']
  )


  #We pad in Hindi Sent
  tgt_padded=tf.keras.preprocessing.sequence.pad_sequences(
      tgt_number,
      padding='post',
      value=hin_vocab['<PAD>']
  )

  return src_padded, tgt_padded

In [ ]:
src_data,tgt_data=prepare_data(data,eng_vocab,hin_vocab)

In [ ]:
src_data

array([[ 1,  3,  4,  5,  2,  0],
       [ 1,  6,  7,  8,  2,  0],
       [ 1,  9,  7, 10,  2,  0],
       [ 1, 11, 12, 13,  2,  0],
       [ 1, 14, 12, 15,  2,  0],
       [ 1, 16,  7, 17,  2,  0],
       [ 1, 18, 12, 19,  2,  0],
       [ 1, 20,  7, 21, 22,  2],
       [ 1, 23,  7, 24, 22,  2],
       [ 1, 25, 12, 26, 22,  2]], dtype=int32)

In [ ]:
tgt_data

array([[ 1,  3,  4,  5,  2,  0],
       [ 1,  6,  7,  8,  2,  0],
       [ 1,  9, 10, 11, 12,  2],
       [ 1, 13, 14, 15, 16,  2],
       [ 1, 13, 17, 18, 16,  2],
       [ 1, 19, 20, 11, 12,  2],
       [ 1, 21,  8, 18, 16,  2],
       [ 1, 22, 23, 12, 24,  2],
       [ 1,  9, 25, 12, 24,  2],
       [ 1, 26, 27, 16, 24,  2]], dtype=int32)

**Encode Arch:**


1. **Embeddeing Layer:** Convert the word index into dense vectoe(list of decimal numbers)

2. **LSTM Layers:** Read the words vector one by one and produce the **Context Vector.**

**Output:**
**Hidden State(ht):** This is a short term memory

**Cell State(ct):** The Long Term Memory

**Together(h,c)==Context Vector**

In [ ]:
class Encoder(tf.keras.Model):
  def __init__(self,vocab_size,embed_size,hidden_size):
    #Vocab Size--> How many words are in eng vocab-->20
    #Embed_size-->size of each word vector(dim)-->50
    #hidden_size-->Size of Context Vector-->100
    super(Encoder,self).__init__()  #it call the parent class constructor

    #Layer1: Embedding Layer
    self.embedding=tf.keras.layers.Embedding(vocab_size,embed_size)

    #Layer2: LSTM Layer
    self.lstm=tf.keras.layers.LSTM(hidden_size,return_state=True)   #it will give me hidden +cell state

  def call(self,x):
    embedded=self.embedding(x)

    _,hidden,cell=self.lstm(embedded)

    return hidden,cell  #context vector

**Decoder Arch:**

It takes CV from Encoder-->It Generates Hindi Translation one word at a time.

**It will have 3 layers:**



*   **Embedding Layer:** Convert Hindi word-->Vector

*   **LSTM Layer**: Generate the next hidden state based in current word+Previous State

* Dense(Fully Connected NN): Convert the LSTM Output to score for Hindi Word:

The Word with the highest score is the predicteed word.



In [ ]:
class Decoder(tf.keras.Model):
  def __init__(self,vocab_size,embed_size,hidden_size):
    super(Decoder,self).__init__()

    #Layer1: Embedding
    self.embedding=tf.keras.layers.Embedding(vocab_size,embed_size)
    #return_sequences=It gives output each time step not just final lstm

    #Layer 2: LSTM
    self.lstm=tf.keras.layers.LSTM(hidden_size,return_state=True,return_sequences=True)

    #Layer3: Dense Layer
    self.dense=tf.keras.layers.Dense(vocab_size)

  def call(self,x,hidden,cell):
    #Step1: Embeddding
    embedded=self.embedding(x)
    #Step2: Pass throught LSTM with the previous state
    lstm_output,hidden,cell=self.lstm(embedded,initial_state=[hidden,cell])
    #Step3: Convert LSTM Output to word score
    output=self.dense(lstm_output)

    return output,hidden,cell

# **Complete Seq2Seq Model Architrcture**

We will combine both Part:

Encoder

Decoder

Training:

During Training, the decoder loop run for each word in the target(hindi) sentence,

At each step:

* It predict a word

* Feed the prediction as input to the next step

* We compare the prediction with actual hindi words to calculte the error(Loss)

* We adjust the model wight to reduce the Loss.

In [ ]:
class Seq2Seq(tf.keras.Model):
  def __init__(self,encoder,decoder,tgt_vocab_size):
    #enocder=object of Encoder
    #decoder=object off Decoder
    #tgt_vocab_size=It is the size of the hindi vocab

    super(Seq2Seq,self).__init__()

    self.encoder=encoder
    self.decoder=decoder
    self.tgt_vocab_size=tgt_vocab_size

  def call(self,inputs,training=False):
    src=inputs[0]  #eng Sent
    tgt=inputs[1]  #Hindi Sent

    #Get batch size and thee target sentence length
    batch_size=tf.shape(src)[0]    #Shape of the eng sentence
    tgt_len=tf.shape(tgt)[1]  #How long is the hindi sentence

    #====================Step 1: Encoder=====================
    hidden,cell=self.encoder(src)
    #====================Step 2: Input word=====================
    input_token=tgt[:,0:1]  #this will get the first column  sos for all the sentence

    #List to  store output at each step
    all_outputs=[]

    #====================Step 3: Decoder Loop===================
    for i in range(1,tgt_len):
      output,hidden,cell=self.decoder(input_token,hidden,cell)
      all_outputs.append(output)
      #We use the predicted word as a input for the next step
      #argmax pick the word with the highest score
      input_token=tf.argmax(output,axis=-1,output_type=tf.int32)   #This is goiing to give the word with the highest scre

  #====================Step4: Combine all Predictions=============
    if all_outputs:
     return tf.concat(all_outputs,axis=1)
    else:
     return tf.zero((batch_size,1,self.tgt_vocab_size))

**Training Function:**

1. Forward Pass: Model Predict Hindi Words.

2. Loss Calcualtion: Compare the prediction with actual hindi words

Use: SparseCategoricalCrossEntopy

This meausred how wrong the predictions are

3. Backward Pass: Calculate Gradients(How to adjust weights).

4. Update Weights: Use Adam Optimizer to adjust wights

5. Repeat the same this for many epochs untill loss value is low.

In [ ]:
def train(model,src_data,tgt_dat,epochs=100):
  optimizer=tf.keras.optimizers.Adam(learning_rate=0.01)


  #loss function
  loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(
      from_logits=True,  #Sincee our model output is raw score
      reduction='none'
  )


  def train_step(src,tgt):
    #GradientTape
    with tf.GradientTape() as tape:
      #Forward Pass:It will get models predicition
      predictions=model([src,tgt],training=True)

      #we are predciting 1 word at a time(1,2,3,4)-->We are not prediction for 0--><SOS>
      target_labels=tgt[:,1:] #Remove <SOS> from target

      #Make sure predcition and targets have same length
      seq_len=target_labels.shape[1]
      predcitions=predictions[:,:seq_len,:]

      #We will create a mask: 1 for real words and 0 for pad tokens
      mask=tf.cast(target_labels!=hin_vocab["<PAD>"],tf.float32)

      #Calculate the loss for each word
      loss=loss_fn(target_labels,predictions)

      #Apply Mask--> Multiply 0 to ignore pad tokens
      loss=loss*mask

      #now only real words we will  calculate the average loss
      total_loss=tf.reduce_sum(loss)   #sum of all the losses
      total_token=tf.reduce_sum(mask)  #count of all the real words
      mean_loss=total_loss/total_token

    #Backward Pass
    gradients=tape.gradient(mean_loss,model.trainable_variables)

    optimizer.apply_gradients(zip(gradients,model.trainable_variables))

    return mean_loss

  print('Training Loop')
  for epoch in range(epochs):
    loss=train_step(src_data,tgt_data)

    if epoch%10==0:
      print(f'Epoch {epoch} | Loss: {loss.numpy():.4f}')

**Now we will define the hyperparameters for my model**


In [ ]:
embed_size=50
hidden_size=100
input_vocab_size=len(eng_vocab)
output_vocab_size=len(hin_vocab)

#Create the Model
encoder=Encoder(input_vocab_size,embed_size,hidden_size)
decoder=Decoder(output_vocab_size,embed_size,hidden_size)


model=Seq2Seq(encoder,decoder,tgt_vocab_size=output_vocab_size)

#Initialize
dummy_src=tf.zeros((1,5),dtype=tf.int32)
dummy_tgt=tf.zeros((1,5),dtype=tf.int32)
_=model([dummy_src,dummy_tgt],training=False)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:1505: UserWarning: Layer 'seq2_seq' looks like it has unbuilt state, but Keras is not able to trace the layer `call()` in order to build it automatically. Possible causes:
1. The `call()` method of your layer may be crashing. Try to `__call__()` the layer eagerly on some test input first to see if it works. E.g. `x = np.random.random((3, 4)); y = layer(x)`
2. If the `call()` method is correct, then you may need to implement the `def build(self, input_shape)` method on your layer. It should create all variables used by the layer (e.g. by calling `layer.build()` on all its children layers).
Exception encountered: '''SymbolicTensor' object cannot be interpreted as an integer''
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'seq2_seq', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This 

In [ ]:
prediction=model([src_data,tgt_data])
prediction

<tf.Tensor: shape=(10, 5, 28), dtype=float32, numpy=
array([[[ 2.4733115e-03,  5.2560307e-04,  8.1531191e-03, ...,
         -1.6507383e-03, -2.3825066e-03,  1.8733630e-03],
        [-1.7675238e-03,  9.9484622e-04,  3.1848387e-03, ...,
         -1.8334293e-03, -3.2693094e-03, -3.1111646e-04],
        [-3.7437356e-03, -1.1766363e-03, -1.8172581e-03, ...,
          5.2475785e-03,  3.1967103e-03, -1.5469326e-03],
        [-6.2673967e-03, -3.2003026e-03, -5.8198324e-03, ...,
          1.0902630e-02,  8.0611529e-03, -2.3027861e-03],
        [-8.8585829e-03, -5.0940453e-03, -8.9863138e-03, ...,
          1.5377217e-02,  1.1763835e-02, -2.7043261e-03]],

       [[ 5.2481331e-03,  1.4036814e-03,  7.0232712e-03, ...,
          2.6796621e-03,  3.1373187e-03, -1.8722356e-03],
        [ 2.8801974e-04,  2.1686375e-03,  2.6962799e-03, ...,
          1.4704242e-03,  9.6237974e-04, -2.7018678e-03],
        [-2.1643443e-03, -1.0294723e-05, -1.9009772e-03, ...,
          7.7401153e-03,  6.4769820e-03, -3

In [ ]:
prediction.shape

TensorShape([10, 5, 28])

In [ ]:
train(model,src_data,tgt_data,epochs=200)

Training Loop
Epoch 0 | Loss: 3.3313
Epoch 10 | Loss: 2.0020
Epoch 20 | Loss: 0.8105
Epoch 30 | Loss: 0.2212
Epoch 40 | Loss: 0.0465
Epoch 50 | Loss: 0.0145
Epoch 60 | Loss: 0.0071
Epoch 70 | Loss: 0.0046
Epoch 80 | Loss: 0.0035
Epoch 90 | Loss: 0.0029
Epoch 100 | Loss: 0.0025
Epoch 110 | Loss: 0.0022
Epoch 120 | Loss: 0.0019
Epoch 130 | Loss: 0.0017
Epoch 140 | Loss: 0.0016
Epoch 150 | Loss: 0.0014
Epoch 160 | Loss: 0.0013
Epoch 170 | Loss: 0.0012
Epoch 180 | Loss: 0.0011
Epoch 190 | Loss: 0.0010


1. tokenize: Conveertinng english sentence into number.

2. Encoder: Pass the input through encoder to get the context vector.

3. Decoder Loop:

  Start with <SOS>
  Predict the next word
  If predcited word is <EOS>-->Stop
  Otherwise-->Feed Predicted word back Repeat.

4. Convert: Change predicted numbers back into hindi words.

In [ ]:
def translation(model,sentence,eng_vocab,hin_vocab,max_len=15):
  #we need to create reverse hindi vecab: Number-->Word(So tht we can convert the predcition back)
  inv_hin={number:word for word,number in hin_vocab.items()}

  #Step1: Tokenize
  tokens=[eng_vocab['<SOS>']] #We will start with <SOS>
  tokens+=[eng_vocab.get(word,0) for word in sentence.split()]  #add word numbers
  tokens+=[eng_vocab['<EOS>']]  #END

  #Need to convert to tenserflow tennsors
  src_tensor=tf.convert_to_tensor([[hin_vocab['<SOS>']]],dtype=tf.int32)

  ##Step2: Encoder:
  hidden,cell=model.encoder(src_tensor)   #We get context  vector

  ##Step3: Decoder Loop:
  #Start with SOS
  input_token=tf.convert_to_tensor([[hin_vocab['<SOS>']]],dtype=tf.int32)

  translated_word=[]   #Empty list

  for i in range(max_len):
    output,hidden,cell=model.decoder(input_token,hidden,cell)

    #pick the word with the highest score(argmax)
    input_token=tf.argmax(output,axis=-1,output_type=tf.int32)

    #Convert tensor to a regural python number
    predicted_index=input_token.numpy()[0,0]

    if predicted_index==hin_vocab['<EOS>']:
      break

    predicted_word=inv_hin.get(predicted_index,'<UNK>')
    translated_word.append(predicted_word)

  return " ".join(translated_word)

In [ ]:
data_dict=dict(data)
data_dict
test_sentence=[
'I am happy',
'We are learning',
'She is singing',
'We are beautiful',
'I love Eating'
]

In [ ]:
for sentence in test_sentence:
  expected=data_dict.get(sentence,'Not in the Training Data')

  #Run my Model
  predicted=translation(model,sentence,eng_vocab,hin_vocab)

  print(f'Input: {sentence}')
  print(f'Expected: {expected}')
  print(f'Predicted: {predicted}')

  #checking
  if expected==predicted:
    print(f'Result was correct')
  else:
    print('Result was wrong')

  print('=' *50)

Input: I am happy
Expected: मैं खुश हूँ
Predicted: तुम सुंदर
Result was wrong
Input: We are learning
Expected: हम सीख रहे हैं
Predicted: तुम सुंदर
Result was wrong
Input: She is singing
Expected: वह गा रही है
Predicted: तुम सुंदर
Result was wrong
Input: We are beautiful
Expected: Not in the Training Data
Predicted: तुम सुंदर
Result was wrong
Input: I love Eating
Expected: Not in the Training Data
Predicted: तुम सुंदर
Result was wrong
